# 22. Chap2-2 보정실험 설계와 데이터셋 생성

이 노트북은 기존 Chapter 2의 confounding 문제를 제거하기 위해 matched factorial 데이터셋을 새로 생성합니다.

설계 원칙:

- color, defect type, shape를 full factorial로 교차합니다.
- matched split에서는 material, camera angle, glare 조건을 모든 color에 동일하게 둡니다.
- stress split에서는 rough material, steep angle, high glare를 모든 color에 동일하게 교차합니다.
- train에는 `neutral`, `blue`, `green`만 넣고, `red`, `purple`은 clean holdout color로 둡니다.
- exposure ratio 실험용 `red scratch` target pool은 eval과 분리해서 따로 생성합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-2장/ch2_2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2*/chap2_2/ch2_2_utils.py"))
        + list(Path.cwd().glob("**/ch2_2_utils.py"))
    )
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "2-2장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch2_2_utils import *

paths = find_ch2_2_paths()
set_korean_font()
set_seed(7)
paths

Chapter22Paths(chap2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), chapter2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-1장'), chapter1_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/1장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/data/synthetic_metal_matched'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs/manifests'))

## 22-1. 데이터셋 생성

In [2]:
DATASET_PARAMS = {
    "image_size": 128,
    "train_per_cell": 12,
    "eval_per_cell": 6,
    "stress_per_cell": 4,
    "target_pool_per_cell": 24,
    "seed": 2207,
    "overwrite": False,
}

data_root = generate_matched_factorial_dataset(paths.data_root, **DATASET_PARAMS)
samples = load_ch2_2_samples(data_root)
print("data_root:", data_root)
print("num_samples:", len(samples))
display(samples[["sample_id", "split", "domain_block", "color_group", "shape_group", "defect_type"]].head())

data_root: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\data\synthetic_metal_matched
num_samples: 736


,sample_id,split,domain_block,color_group,shape_group,defect_type
0,train_matched_matched_control_neutral_scratch_...,train_matched,matched_control,neutral,top_half_metal,scratch
1,train_matched_matched_control_neutral_scratch_...,train_matched,matched_control,neutral,top_half_metal,scratch
2,train_matched_matched_control_neutral_scratch_...,train_matched,matched_control,neutral,top_half_metal,scratch
3,train_matched_matched_control_neutral_scratch_...,train_matched,matched_control,neutral,top_half_metal,scratch
4,train_matched_matched_control_neutral_scratch_...,train_matched,matched_control,neutral,top_half_metal,scratch


## 22-2. 설계 파일 저장

In [3]:
design = {
    "purpose": "Chapter 2-2 matched factorial correction for SegFormer generalization experiments",
    "seen_colors": SEEN_COLORS,
    "heldout_colors": HELDOUT_COLORS,
    "all_colors": ALL_COLORS,
    "shape_groups": SHAPE_GROUPS,
    "defect_groups": DEFECT_GROUPS,
    "dataset_params": DATASET_PARAMS,
    "validity_rule": [
        "Only one factor should change in factor-specific comparisons.",
        "Material, camera angle, and glare are matched inside eval_matched.",
        "Stress factors are crossed separately in eval_stress.",
        "Main conclusions must use repeated model seeds.",
    ],
}
save_json(paths.runs_root / "chapter2_2_design_plan.json", design)
print(paths.runs_root / "chapter2_2_design_plan.json")

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\chapter2_2_design_plan.json


## 22-3. Manifest 생성과 빠른 분포 확인

In [4]:
manifests = create_ch2_2_manifests(samples, paths.runs_root)
for name, path in manifests.items():
    print(f"{name:16s}", path)

count_table = (
    samples.groupby(["split", "color_group", "shape_group", "defect_type"])
    .size()
    .reset_index(name="count")
)
display(count_table.head(30))
display(samples.groupby(["split", "color_group"]).size().reset_index(name="count"))

train            C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\train_manifest.csv
target_pool      C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\target_pool_manifest.csv
eval_matched     C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\eval_matched_manifest.csv
eval_stress      C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\eval_stress_manifest.csv


,split,color_group,shape_group,defect_type,count
0,eval_matched,blue,bottom_half_metal,dent,6
1,eval_matched,blue,bottom_half_metal,impact,6
2,eval_matched,blue,bottom_half_metal,scratch,6
3,eval_matched,blue,bottom_half_metal,stain,6
4,eval_matched,blue,top_half_metal,dent,6
5,eval_matched,blue,top_half_metal,impact,6
6,eval_matched,blue,top_half_metal,scratch,6
7,eval_matched,blue,top_half_metal,stain,6
8,eval_matched,green,bottom_half_metal,dent,6
9,eval_matched,green,bottom_half_metal,impact,6


,split,color_group,count
0,eval_matched,blue,48
1,eval_matched,green,48
2,eval_matched,neutral,48
3,eval_matched,purple,48
4,eval_matched,red,48
5,eval_stress,blue,32
6,eval_stress,green,32
7,eval_stress,neutral,32
8,eval_stress,purple,32
9,eval_stress,red,32


## 22-4. 이 노트북의 산출물

In [5]:
print("dataset:", paths.data_root)
print("runs:", paths.runs_root)
print("next notebook: 23_Chap2_2_Confounding_Audit.ipynb")

dataset: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\data\synthetic_metal_matched
runs: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs
next notebook: 23_Chap2_2_Confounding_Audit.ipynb
